# Cierre seguro de pendientes — Urban-v1

Este Notebook reúne en una sola ejecución:

1. Congelar y hashear la configuración efectiva derivada del protocolo.
2. Declarar y verificar la procedencia de pesos (inicialización aleatoria; sin pesos preentrenados ni checkpoint).
3. Auditar manifiestos e índices completos de TRAIN/VAL, soporte por clase e inspección visual reproducible.

**Seguridad:** `RUN_TRAINING=False`. No descarga datasets, no ejecuta forward/backward de entrenamiento, no llama `optimizer.step()`, no escribe checkpoints, no abre `test` y no lanza CI. La inspección visual solo abre una muestra determinista de imágenes/etiquetas TRAIN/VAL ya presentes en Drive.

El resultado final se guarda en Drive y la última celda imprime el único bloque que debes pegar aquí.

## 1. Preparar Drive y sincronizar el repositorio

En Colab, configura el secreto `GITHUB_TOKEN` con permiso **Contents: Read-only**. No pegues tokens en el notebook ni en el chat.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys, json, hashlib, datetime, platform, subprocess, getpass, random, textwrap
from PIL import Image, ImageDraw
import numpy as np

RUN_TRAINING = False
assert RUN_TRAINING is False
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

REPO='Leohf04/Procesamiento-de-Im-genes-'
BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try:
    TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN=None
if not TOKEN:
    TOKEN=getpass.getpass('GitHub PAT (Contents: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta GITHUB_TOKEN.')

askpass=Path('/tmp/urban_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) echo x-access-token ;;\n *Password*) printf "%s\\n" "$COLAB_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env=os.environ.copy()
env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
del TOKEN

def git(args):
    p=subprocess.run(['git']+args,capture_output=True,text=True,env=env)
    if p.returncode:
        raise RuntimeError('git '+ ' '.join(args)+'\n'+(p.stderr or p.stdout)[-4000:])
    return p.stdout.strip()

if WORKDIR.exists() and (WORKDIR/'.git').exists():
    git(['-C',str(WORKDIR),'remote','set-url','origin',f'https://github.com/{REPO}.git'])
    git(['-C',str(WORKDIR),'fetch','origin',BRANCH])
    git(['-C',str(WORKDIR),'checkout',BRANCH])
    git(['-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH])
elif WORKDIR.exists():
    raise RuntimeError(f'{WORKDIR} existe y no es repo Git; no se sobrescribe.')
else:
    git(['clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)])
env.pop('COLAB_GITHUB_TOKEN',None); env.pop('GIT_ASKPASS',None)
os.chdir(WORKDIR)
sys.path.insert(0,str(WORKDIR))
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
DRIVE_ROOT=Path('/content/drive/MyDrive/ProcesamientoImagenes')
DATA_DIR=DRIVE_ROOT/'urban_data_capacity'
MANIFEST_DIR=DATA_DIR/'local_manifests'
INDEX_DIR=DATA_DIR/'local_annotation_index'
OUT_DIR=DRIVE_ROOT/'urban_v1_closure_audit'
OUT_DIR.mkdir(parents=True,exist_ok=True)
print('HEAD:',HEAD)
print('RUN_TRAINING:',RUN_TRAINING)
print('TRAIN/VAL only; test access disabled.')

## 2. Congelar configuración efectiva y procedencia de pesos

La configuración se deriva literalmente de la tabla de `urban-exp-v1`. Se registra como propuesta reproducible, no como autorización de entrenamiento. La inicialización aleatoria queda declarada porque la factoría actual usa `pretrained_backbone=False` y `weights=None`; no se afirma que existan pesos preentrenados.

In [ ]:
from pathlib import Path
config_path=WORKDIR/'configs/urban_experiment_v1.json'
config=json.loads(config_path.read_text(encoding='utf-8'))
ontology=json.loads((WORKDIR/config['ontology']).read_text(encoding='utf-8'))
assert config['ontology_version']==ontology['ontology_version']=='urban-v1'
assert config['run_policy']['RUN_TRAINING'] is False
assert config['initialization']['pretrained_weights_used'] is False
assert config['initialization']['checkpoint_loaded'] is False
assert len(config['experiments'])==4

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

config_sha=sha256_file(config_path)
ontology_sha=sha256_file(WORKDIR/config['ontology'])
factory=(WORKDIR/'scripts/urban_experiment_factory.py').read_text(encoding='utf-8')
runner_source=(WORKDIR/'src/urban_experiment_runner.py').read_text(encoding='utf-8')
optimizer_source=(WORKDIR/'src/urban_optimizer.py').read_text(encoding='utf-8')
factory_configs=[]
required={'dataset','task','image_size','batch_size','learning_rate','experiment_id',
          'max_epochs','patience','max_grad_norm','gradient_accumulation_steps',
          'scheduler_factor','scheduler_patience'}
for exp in config['experiments']:
    runtime_config=dict(exp)
    runtime_config['data_dir']=str(DATA_DIR)
    runtime_config['annotation_index_dir']=str(INDEX_DIR)
    missing=required-set(runtime_config)
    if missing:
        raise AssertionError(f"{exp['id']} missing factory/runner fields: {sorted(missing)}")
    factory_configs.append(runtime_config)

provenance_checks={
 'semantic_random_init_declared':'pretrained_backbone=False' in factory,
 'instances_random_init_declared':'weights=None,weights_backbone=None' in factory.replace(' ','').replace('\\n',''),
 'gradient_accumulation_implemented':'gradient_accumulation_steps' in runner_source,
 'scheduler_factor_implemented':'factor=float(config.get("scheduler_factor", 0.5))' in optimizer_source,
 'all_four_factory_configs_complete':len(factory_configs)==4,
 'no_checkpoint_load':config['initialization']['checkpoint_loaded'] is False,
 'training_disabled':RUN_TRAINING is False
}
if not all(provenance_checks.values()):
    raise AssertionError(f'No se pudo verificar configuración/procedencia: {provenance_checks}')
source_hashes={
 'scripts/urban_experiment_factory.py':sha256_file(WORKDIR/'scripts/urban_experiment_factory.py'),
 'src/urban_experiment_runner.py':sha256_file(WORKDIR/'src/urban_experiment_runner.py'),
 'src/urban_optimizer.py':sha256_file(WORKDIR/'src/urban_optimizer.py')
}
print('CONFIG SHA256:',config_sha)
print('ONTOLOGY SHA256:',ontology_sha)
print('PROCEDENCIA/IMPLEMENTACIÓN:',json.dumps(provenance_checks,indent=2))
print('Experimentos congelados:',[x['id'] for x in config['experiments']])
print('Hash de código:',json.dumps(source_hashes,indent=2))
print('Nota: batch 4 semántico y acumulación 2 instancias requieren piloto autorizado; no se prueban aquí.')

## 3. Auditoría completa de manifiestos e índices TRAIN/VAL

Se comparan IDs, orden, unicidad, estados y existencia de assets listados. Se calculan soportes semánticos y de instancias a partir de índices completos. No se accede a carpetas ni manifiestos `test`.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

def asset_candidates(value):
    """Resolve legacy Colab paths against persistent Drive without hiding missing files."""
    if not isinstance(value, str) or not value.strip():
        return []
    raw = Path(value.strip())
    candidates = []
    def add(path):
        path = Path(path)
        if path not in candidates:
            candidates.append(path)

    add(raw)
    legacy_root = Path('/content/urban_data_capacity')
    try:
        rel = raw.relative_to(legacy_root)
        add(DATA_DIR / rel)
    except ValueError:
        pass

    # Also handle legacy paths rooted at /content/ProcesamientoImagenes or relative paths.
    for prefix in (Path('/content/drive/MyDrive/ProcesamientoImagenes/urban_data_capacity'),
                   Path('/content/ProcesamientoImagenes/urban_data_capacity')):
        try:
            rel = raw.relative_to(prefix)
            add(DATA_DIR / rel)
        except ValueError:
            pass
    if not raw.is_absolute():
        add(DATA_DIR / raw)
        add(WORKDIR / raw)
        add(Path('/content') / raw)
    return candidates

def resolve_asset(value):
    candidates = asset_candidates(value)
    return next((p for p in candidates if p.is_file()), None)

def asset_resolution(value):
    candidates = asset_candidates(value)
    found = next((p for p in candidates if p.is_file()), None)
    original = Path(value) if isinstance(value, str) and value.strip() else None
    if found is None:
        kind = 'not_materialized_or_unavailable'
    elif original is not None and found == original:
        kind = 'original_path'
    elif original is not None and str(original).startswith('/content/urban_data_capacity/'):
        kind = 'legacy_path_remapped_to_drive'
    else:
        kind = 'alternate_candidate'
    return {
        'found': found is not None,
        'resolved_path': str(found) if found else None,
        'resolution_kind': kind,
        'checked_candidates': [str(p) for p in candidates[:5]],
    }

# Conteos completos esperados definidos por el protocolo del proyecto.
expected_rows = {
    'a2d2': {'train': 1000, 'val': 1000},
    'urbansyn': {'train': 1000, 'val': 751},
}

audit_data = {}
manifest_rows_cache = {}
asset_resolution_totals = {
    'fields_checked': 0, 'found_at_original_path': 0,
    'found_after_path_remap': 0, 'not_found': 0,
    'unique_files_found': 0, 'unique_files_missing': 0,
}
manifest_source_diagnostics = {}

for ds in ('a2d2', 'urbansyn'):
    for split in ('train', 'val'):
        mp = MANIFEST_DIR / f'{ds}_{split}.jsonl'
        ip = INDEX_DIR / f'{ds}_{split}.annotations.jsonl'
        key = f'{ds}_{split}'
        item = {'manifest_path': str(mp), 'index_path': str(ip), 'errors': []}

        # Diagnosticar explícitamente los ficheros fuente configurados, sin recorrer test.
        manifest_source_diagnostics[key] = {
            'configured_manifest_exists': mp.is_file(),
            'configured_index_exists': ip.is_file(),
            'expected_rows': expected_rows[ds][split],
            'manifest_candidates_checked': [str(mp), str(DATA_DIR / 'manifests' / mp.name),
                                            str(DRIVE_ROOT / 'manifests' / mp.name),
                                            str(WORKDIR / 'artifacts' / 'manifests' / mp.name)],
        }
        if not mp.is_file() or not ip.is_file():
            item['errors'].append('manifest_or_index_missing')
            audit_data[key] = item
            continue

        manifest = read_jsonl(mp)
        index = read_jsonl(ip)
        manifest_rows_cache[key] = manifest
        mids = [str(row.get('sample_id')) for row in manifest]
        iids = [str(row.get('sample_id')) for row in index]

        item.update({
            'manifest_rows': len(manifest), 'index_rows': len(index),
            'expected_manifest_rows': expected_rows[ds][split],
            'manifest_rows_match_protocol': len(manifest) == expected_rows[ds][split],
            'manifest_sha256': sha256_file(mp), 'index_sha256': sha256_file(ip),
            'manifest_ids_unique': len(mids) == len(set(mids)),
            'index_ids_unique': len(iids) == len(set(iids)),
            'ids_same_order': mids == iids, 'ids_same_set': set(mids) == set(iids),
        })
        if len(manifest) != expected_rows[ds][split]:
            item['errors'].append('manifest_row_count_differs_from_protocol')
        if len(mids) != len(set(mids)):
            item['errors'].append('duplicate_manifest_ids')
        if len(iids) != len(set(iids)):
            item['errors'].append('duplicate_index_ids')
        if set(mids) != set(iids):
            item['errors'].append('manifest_index_id_mismatch')
        if mids != iids:
            item['errors'].append('manifest_index_order_mismatch')

        bad_status = [str(row.get('sample_id')) for row in index
                      if row.get('annotation_status', 'ok') != 'ok']
        item['annotation_status_not_ok_count'] = len(bad_status)
        if bad_status:
            item['errors'].append('annotation_status_not_ok')

        semantic_pixels, instance_counts = {}, {}
        for row in index:
            for class_name, count in (row.get('semantic_pixels') or {}).items():
                semantic_pixels[class_name] = semantic_pixels.get(class_name, 0) + int(count)
            counts = row.get('instance_counts') or row.get('instances_by_class') or {}
            for class_name, count in counts.items():
                instance_counts[class_name] = instance_counts.get(class_name, 0) + int(count)
        item['semantic_pixels'] = semantic_pixels
        item['instance_counts'] = instance_counts

        required_fields = {'a2d2': ('image', 'semantic', 'instance'),
                           'urbansyn': ('rgb', 'semantic', 'panoptic')}[ds]
        missing_required_fields = []
        for row in manifest:
            for field in required_fields:
                value = row.get(field)
                if not isinstance(value, str) or not value.strip():
                    missing_required_fields.append({'sample_id': row.get('sample_id'), 'field': field})
        item['missing_required_fields_count'] = len(missing_required_fields)
        item['missing_required_fields_examples'] = missing_required_fields[:20]
        if missing_required_fields:
            item['errors'].append('required_manifest_fields_missing')

        # Comprobar assets usando tanto la ruta original como su equivalente persistente en Drive.
        asset_fields = ('image', 'rgb', 'image_path', 'rgb_path', 'semantic', 'semantic_path',
                        'instance', 'instance_path', 'panoptic', 'panoptic_path',
                        'label', 'label_path', 'mask', 'mask_path')
        asset_missing, asset_details, unique_found, unique_missing = [], [], set(), set()
        for row in manifest:
            for field in asset_fields:
                value = row.get(field)
                if isinstance(value, str) and value.strip():
                    resolution = asset_resolution(value)
                    asset_resolution_totals['fields_checked'] += 1
                    asset_details.append({'sample_id': row.get('sample_id'), 'field': field,
                                          'original_path': value, **resolution})
                    if resolution['found']:
                        unique_found.add(resolution['resolved_path'])
                        if resolution['resolution_kind'] == 'original_path':
                            asset_resolution_totals['found_at_original_path'] += 1
                        else:
                            asset_resolution_totals['found_after_path_remap'] += 1
                    else:
                        unique_missing.add(value)
                        asset_missing.append({'sample_id': row.get('sample_id'),
                                              'field': field, 'path': value,
                                              'resolution_kind': resolution['resolution_kind']})
        item['missing_assets_count'] = len(asset_missing)
        item['missing_assets_examples'] = asset_missing[:20]
        item['asset_resolution_counts'] = {
            'fields_checked': len(asset_details),
            'found': sum(1 for x in asset_details if x['found']),
            'legacy_paths_remapped': sum(1 for x in asset_details
                                         if x['resolution_kind'] == 'legacy_path_remapped_to_drive'),
            'not_found': sum(1 for x in asset_details if not x['found']),
            'unique_files_found': len(unique_found),
            'unique_file_paths_missing': len(unique_missing),
        }
        item['asset_resolution_examples'] = [
            x for x in asset_details if x['resolution_kind'] != 'original_path'
        ][:12]
        asset_resolution_totals['unique_files_found'] += len(unique_found)
        asset_resolution_totals['unique_files_missing'] += len(unique_missing)
        if asset_missing:
            item['errors'].append('manifest_asset_missing')
        audit_data[key] = item

for ds in ('a2d2', 'urbansyn'):
    train_key, val_key = f'{ds}_train', f'{ds}_val'
    if train_key in manifest_rows_cache and val_key in manifest_rows_cache:
        overlap = sorted({str(x.get('sample_id')) for x in manifest_rows_cache[train_key]} &
                         {str(x.get('sample_id')) for x in manifest_rows_cache[val_key]})
        for key in (train_key, val_key):
            audit_data[key]['train_val_overlap_count'] = len(overlap)
            audit_data[key]['train_val_overlap_examples'] = overlap[:20]
            if overlap and 'train_val_leakage' not in audit_data[key]['errors']:
                audit_data[key]['errors'].append('train_val_leakage')

print('RESOLUCIÓN DE ASSETS:', json.dumps(asset_resolution_totals, indent=2))
print('DIAGNÓSTICO DE MANIFIESTOS:', json.dumps(manifest_source_diagnostics, indent=2))
print(json.dumps({
    key: {field: value for field, value in item.items()
          if field not in ('semantic_pixels', 'instance_counts', 'missing_assets_examples',
                           'asset_resolution_examples')}
    for key, item in audit_data.items()
}, indent=2))


## 4. Inspección visual determinista de TRAIN/VAL

Genera una hoja de contacto privada con hasta 3 muestras por dataset y split (12 como máximo). Muestra RGB y, cuando los manifiestos indiquen rutas válidas, la etiqueta semántica o de instancias. No abre `test`. Revisa la hoja en la salida y anota cualquier problema visual evidente.

In [ ]:
from IPython.display import display
from PIL import ImageOps, ImageDraw
import math

def first_existing(row, keys):
    for key in keys:
        value = row.get(key)
        if isinstance(value, str):
            p = resolve_asset(value)
            if p and p.is_file():
                return p
    return None

tiles = []
visual_report = {}
for ds in ('a2d2', 'urbansyn'):
    for split in ('train', 'val'):
        mp = MANIFEST_DIR / f'{ds}_{split}.jsonl'
        rows = read_jsonl(mp) if mp.is_file() else []
        chosen = []
        if rows:
            idxs = sorted(set([0, len(rows)//2, len(rows)-1]))[:3]
            chosen = [rows[i] for i in idxs]
        visual_report[f'{ds}_{split}'] = []
        for row in chosen:
            sid = str(row.get('sample_id', 'unknown'))
            imgp = first_existing(row, ('image', 'rgb', 'image_path', 'rgb_path'))
            labelp = first_existing(row, ('semantic', 'semantic_path', 'panoptic',
                                          'panoptic_path', 'instance', 'instance_path',
                                          'mask', 'mask_path', 'label', 'label_path'))
            rec = {'sample_id': sid, 'image_found': bool(imgp), 'label_found': bool(labelp),
                   'image_path': str(imgp) if imgp else None,
                   'label_path': str(labelp) if labelp else None}
            visual_report[f'{ds}_{split}'].append(rec)
            if not imgp:
                continue
            im = Image.open(imgp).convert('RGB')
            im.thumbnail((320, 200))
            canvas = Image.new('RGB', (340, 260), 'white')
            canvas.paste(im, ((340-im.width)//2, 28))
            draw = ImageDraw.Draw(canvas)
            draw.text((8, 6), f'{ds}/{split} | {sid[:28]}', fill='black')
            if labelp:
                lab = Image.open(labelp)
                lab.thumbnail((320, 200), Image.Resampling.NEAREST)
                if lab.mode not in ('RGB', 'RGBA'):
                    arr = np.asarray(lab)
                    if arr.ndim == 2:
                        norm = (arr.astype(np.float32) % 256).astype(np.uint8)
                        lab = Image.fromarray(np.stack([norm, np.roll(norm, 70, axis=0),
                                                       np.roll(norm, 130, axis=1)], axis=-1))
                    else:
                        lab = lab.convert('RGB')
                else:
                    lab = lab.convert('RGB')
                lab.thumbnail((140, 90))
                canvas.paste(lab, (190, 165))
            tiles.append(canvas)

if tiles:
    cols = 2
    rows_n = math.ceil(len(tiles)/cols)
    sheet = Image.new('RGB', (cols*340, rows_n*260), (235, 235, 235))
    for i, tile in enumerate(tiles):
        sheet.paste(tile, ((i%cols)*340, (i//cols)*260))
    visual_path = OUT_DIR/'train_val_visual_contact_sheet.jpg'
    sheet.save(visual_path, quality=90)
    display(sheet)
else:
    visual_path = None
    print('No se pudieron construir mosaicos: no se encontraron imágenes TRAIN/VAL.')

print('Hoja visual:', visual_path)
print('Registros visuales:', json.dumps(visual_report, indent=2))


## 5. Consolidar auditoría y comprobar puertas

Un split solo pasa si manifiesto/índice existen, IDs y orden coinciden, no hay duplicados, estados incorrectos, assets faltantes ni fuga TRAIN/VAL. La inspección visual es una ayuda humana: debes revisar la hoja y confirmar si detectas errores evidentes. La configuración y procedencia quedan hasheadas y versionadas.

In [ ]:
all_data_checks = all(
    v.get('errors') == [] and
    v.get('manifest_rows', 0) == v.get('expected_manifest_rows') and
    v.get('index_rows', 0) == v.get('expected_manifest_rows')
    for v in audit_data.values()
) and len(audit_data) == 4
visual_count = sum(len(v) for v in visual_report.values())
visual_images = sum(1 for group in visual_report.values() for x in group if x['image_found'])
summary = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'git_head': HEAD, 'branch': BRANCH, 'python': platform.python_version(),
    'config_path': str(config_path), 'config_sha256': config_sha,
    'ontology_sha256': ontology_sha, 'config': config, 'source_hashes': source_hashes,
    'weight_provenance_checks': provenance_checks,
    'data_audit_train_val_only': audit_data,
    'manifest_source_diagnostics': manifest_source_diagnostics,
    'asset_resolution_summary': asset_resolution_totals,
    'all_data_checks_pass': all_data_checks,
    'visual_report_train_val_only': visual_report,
    'visual_samples_selected': visual_count, 'visual_images_opened': visual_images,
    'visual_contact_sheet': str(visual_path) if visual_path else None,
    'visual_review_user_confirmed': False,
    'RUN_TRAINING': False, 'optimizer_steps': 0, 'backward_calls': 0, 'epochs_started': 0,
    'checkpoint_written': False, 'dataset_downloaded': False, 'test_split_touched': False,
    'experiment_readiness': 'BLOCKED_PENDING_USER_VISUAL_REVIEW_OR_DATA_AUDIT'
        if not all_data_checks else 'BLOCKED_PENDING_USER_VISUAL_REVIEW_AND_PILOT_AUTHORIZATION'
}
summary_path = OUT_DIR/'urban_v1_closure_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Resumen guardado:', summary_path)
print('Auditoría datos:', 'PASS' if all_data_checks else 'INCOMPLETE')
print('Assets:', json.dumps(asset_resolution_totals, indent=2))
print('Imágenes inspeccionadas por notebook:', visual_images)
print('Entrenamiento:', RUN_TRAINING, '| test touched:', False)


## 6. ÚLTIMA CELDA — bloque único para continuar aquí

Ejecuta esta celda al final. Copia todo lo comprendido entre los delimitadores y pégalo en ChatGPT. Incluye hashes, configuración, procedencia, soporte, faltantes, revisión visual y rutas persistentes.

In [ ]:
payload = {
    'status': 'PASS' if all_data_checks and all(provenance_checks.values()) else 'INCOMPLETE',
    'git_head': HEAD, 'branch': BRANCH,
    'config_sha256': config_sha, 'ontology_sha256': ontology_sha,
    'source_hashes': source_hashes, 'config_path': str(config_path),
    'summary_path': str(summary_path),
    'visual_contact_sheet': str(visual_path) if visual_path else None,
    'weight_provenance_checks': provenance_checks,
    'data_audit_train_val_only': audit_data,
    'manifest_source_diagnostics': manifest_source_diagnostics,
    'asset_resolution_summary': asset_resolution_totals,
    'all_data_checks_pass': all_data_checks,
    'visual_report_train_val_only': visual_report,
    'visual_samples_selected': visual_count, 'visual_images_opened': visual_images,
    'visual_review_user_confirmed': False,
    'experiment_readiness': summary['experiment_readiness'],
    'RUN_TRAINING': False, 'optimizer_steps': 0, 'backward_calls': 0,
    'epochs_started': 0, 'checkpoint_written': False,
    'dataset_downloaded': False, 'test_split_touched': False
}
block = '===== BLOQUE_CONTINUIDAD_INICIO =====\n' + json.dumps(
    payload, ensure_ascii=False, sort_keys=True
) + '\n===== BLOQUE_CONTINUIDAD_FIN ====='
print(block)
print('JSON persistido en Drive:', summary_path)
